In [26]:
import pandas as pd
import numpy as np
from numpy import arccos, clip
from scipy.stats import linregress, skew, kurtosis, entropy
from scipy.fft import fft
from functools import reduce

import warnings
warnings.filterwarnings('ignore')


In [27]:
df_original = pd.read_csv('../../data_cleaning/data/imputed_data.csv')
df = df_original.copy()

In [28]:
df.rename(columns={'person': 'participant'}, inplace=True)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.sort_values(by=['timestamp'], ascending=True, inplace=True)
df

,timestamp,heart_rate,x_coordinate,y_coordinate,pupil_diameter_mm,iris_diameter_mm,pupil_iris_ratio,genre,participant
6570,2025-06-05 16:25:55,62.0,409.206897,194.000000,2.602941,11.8,0.220588,comedy,kenji
6571,2025-06-05 16:25:56,61.5,413.433333,194.800000,2.602941,11.8,0.220588,comedy,kenji
6572,2025-06-05 16:25:59,60.0,397.333333,191.000000,2.602941,11.8,0.220588,comedy,kenji
6573,2025-06-05 16:26:01,60.0,397.166667,190.600000,2.602941,11.8,0.220588,comedy,kenji
6574,2025-06-05 16:26:04,60.0,391.000000,189.333333,2.602941,11.8,0.220588,comedy,kenji
...,...,...,...,...,...,...,...,...,...
2449,2025-06-12 10:33:07,70.0,356.000000,324.000000,2.726626,11.8,0.231070,horror,aimen
2450,2025-06-12 10:33:08,70.0,356.000000,324.000000,2.726626,11.8,0.231070,horror,aimen
2451,2025-06-12 10:33:09,70.0,355.741935,324.000000,2.726626,11.8,0.231070,horror,aimen
2452,2025-06-12 10:33:10,70.0,356.000000,324.000000,2.726626,11.8,0.231070,horror,aimen


In [29]:
window_size = 2

## 1. Head movement features

In [30]:
# Head displacement per frame
df['dx'] = df.groupby(['participant', 'genre'])['x_coordinate'].diff()
df['dy'] = df.groupby(['participant', 'genre'])['y_coordinate'].diff()
df['head_displacement'] = np.sqrt(df['dx']**2 + df['dy']**2)
df['head_displacement'] = df['head_displacement'].fillna(0)

# Head velocity per frame
df['dt'] = df.groupby(['participant', 'genre'])['timestamp'].diff().dt.total_seconds()
df['head_velocity'] = df['head_displacement'] / df['dt']
df['head_velocity'] = df['head_velocity'].fillna(0)

# Head direction change rate
df['dx_prev'] = df.groupby(['participant', 'genre'])['dx'].shift()
df['dy_prev'] = df.groupby(['participant', 'genre'])['dy'].shift()

dot = df['dx'] * df['dx_prev'] + df['dy'] * df['dy_prev']
norm_product = np.sqrt((df['dx']**2 + df['dy']**2) * (df['dx_prev']**2 + df['dy_prev']**2))
df['cos_theta'] = dot / norm_product
df['cos_theta'] = clip(df['cos_theta'], -1.0, 1.0)  # Clip for numeric stability

df['angle_change'] = arccos(df['cos_theta'])

df['head_direction_change_rate'] = df.groupby(['participant', 'genre'])['angle_change'].rolling(window=window_size, min_periods=1).mean().reset_index(level=[0,1], drop=True)
df['head_direction_change_rate'] = df['head_direction_change_rate'].fillna(0)

# Head stability
df['head_x_std'] = df.groupby(['participant', 'genre'])['x_coordinate'].rolling(window=window_size, min_periods=1).std().reset_index(level=[0,1], drop=True)
df['head_y_std'] = df.groupby(['participant', 'genre'])['y_coordinate'].rolling(window=window_size, min_periods=1).std().reset_index(level=[0,1], drop=True)
df['head_stability'] = (df['head_x_std'] + df['head_y_std']) / 2
df['head_stability'] = df['head_stability'].fillna(0)

# Centered coordinates
mean_x = df.groupby(['participant', 'genre'])['x_coordinate'].transform('mean')
mean_y = df.groupby(['participant', 'genre'])['y_coordinate'].transform('mean')
df['centered_x'] = df['x_coordinate'] - mean_x
df['centered_y'] = df['y_coordinate'] - mean_y

# Drop intermediate columns if needed
df.drop(columns=['dx', 'dy', 'dt', 'dx_prev', 'dy_prev', 'cos_theta', 'angle_change', 'head_x_std', 'head_y_std'], inplace=True)

In [31]:
def reorder_cols(df): 
    df.sort_values(by=['timestamp'], ascending=True, inplace=True)

    # Get the current column order
    cols = list(df.columns)
    
    # Move 'participant' to 2nd and 'genre' to last
    cols.remove('participant')
    cols.remove('genre')
    new_order = [cols[0], 'participant'] + cols[1:] + ['genre']
    
    # Reorder the DataFrame
    return df[new_order]

df = reorder_cols(df)
df

,timestamp,participant,heart_rate,x_coordinate,y_coordinate,pupil_diameter_mm,iris_diameter_mm,pupil_iris_ratio,head_displacement,head_velocity,head_direction_change_rate,head_stability,centered_x,centered_y,genre
6570,2025-06-05 16:25:55,kenji,62.0,409.206897,194.000000,2.602941,11.8,0.220588,0.000000,0.000000,0.000000,0.000000,33.476335,-15.891058,comedy
6571,2025-06-05 16:25:56,kenji,61.5,413.433333,194.800000,2.602941,11.8,0.220588,4.301484,4.301484,0.000000,1.777114,37.702772,-15.091058,comedy
6572,2025-06-05 16:25:59,kenji,60.0,397.333333,191.000000,2.602941,11.8,0.220588,16.542370,5.514123,3.096881,7.035712,21.602772,-18.891058,comedy
6573,2025-06-05 16:26:01,kenji,60.0,397.166667,190.600000,2.602941,11.8,0.220588,0.433333,0.216667,2.020552,0.200347,21.436105,-19.291058,comedy
6574,2025-06-05 16:26:04,kenji,60.0,391.000000,189.333333,2.602941,11.8,0.220588,6.295413,2.098471,0.958820,2.628080,15.269439,-20.557725,comedy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2449,2025-06-12 10:33:07,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.509902,0.509902,0.184065,0.212132,-64.809663,139.658730,horror
2450,2025-06-12 10:33:08,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.000000,0.000000,0.259385,0.000000,-64.809663,139.658730,horror
2451,2025-06-12 10:33:09,aimen,70.0,355.741935,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,0.000000,0.091240,-65.067728,139.658730,horror
2452,2025-06-12 10:33:10,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,3.141593,0.091240,-64.809663,139.658730,horror


## 2. Time Domain Features

In [32]:
# Define features and window size
signals = ['heart_rate', 'centered_x', 'centered_y', 'pupil_diameter_mm', 'iris_diameter_mm', 'pupil_iris_ratio']
window = str(window_size)+ 's'

# Ensure timestamp is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Function for custom features
def compute_features(group):
    group = group.set_index('timestamp').sort_index()

    for signal in signals:
        roll = group[signal].rolling(window=window, min_periods=2)

        # Basic stats
        group[f'td_{signal}_mean'] = roll.mean()
        group[f'td_{signal}_std'] = roll.std()
        group[f'td_{signal}_min'] = roll.min()
        group[f'td_{signal}_max'] = roll.max()
        group[f'td_{signal}_range'] = group[f'td_{signal}_max'] - group[f'td_{signal}_min']

        # Slope (linear regression over rolling window)
        group[f'td_{signal}_slope'] = roll.apply(
            lambda x: linregress(range(len(x)), x)[0] if len(x) > 1 else np.nan, raw=False
        )

        # Mean absolute change
        group[f'td_{signal}_mean_abs_change'] = roll.apply(
            lambda x: np.mean(np.abs(np.diff(x))) if len(x) > 1 else np.nan,
            raw=True
        )

        # Second-order difference mean 
        group[f'td_{signal}_second_order_diff_mean'] = roll.apply(
            lambda x: np.mean(np.abs(np.diff(x, n=2))) if len(x) > 2 else np.nan,
            raw=True
        )

    return group.reset_index()

# Apply per participant per genre
df = df.groupby(['participant', 'genre'], group_keys=False).apply(compute_features)

In [33]:
df.sort_values(by=['timestamp'], ascending=True, inplace=True)

In [34]:
# Backward fill because first one or two rows could be NaN
df = reorder_cols(df)
td_feature_columns = [col for col in df.columns if col.startswith('td_')]
df[td_feature_columns] = df[td_feature_columns].fillna(method='bfill').fillna(0)

In [35]:
df

,timestamp,participant,heart_rate,x_coordinate,y_coordinate,pupil_diameter_mm,iris_diameter_mm,pupil_iris_ratio,head_displacement,head_velocity,...,td_iris_diameter_mm_second_order_diff_mean,td_pupil_iris_ratio_mean,td_pupil_iris_ratio_std,td_pupil_iris_ratio_min,td_pupil_iris_ratio_max,td_pupil_iris_ratio_range,td_pupil_iris_ratio_slope,td_pupil_iris_ratio_mean_abs_change,td_pupil_iris_ratio_second_order_diff_mean,genre
0,2025-06-05 16:25:55,kenji,62.0,409.206897,194.000000,2.602941,11.8,0.220588,0.000000,0.000000,...,0.0,0.220588,0.000000,0.220588,0.220588,0.000000,0.000000,0.000000,0.000373,comedy
1,2025-06-05 16:25:56,kenji,61.5,413.433333,194.800000,2.602941,11.8,0.220588,4.301484,4.301484,...,0.0,0.220588,0.000000,0.220588,0.220588,0.000000,0.000000,0.000000,0.000373,comedy
2,2025-06-05 16:25:59,kenji,60.0,397.333333,191.000000,2.602941,11.8,0.220588,16.542370,5.514123,...,0.0,0.216246,0.006141,0.211904,0.220588,0.008684,-0.008684,0.008684,0.000373,comedy
3,2025-06-05 16:26:01,kenji,60.0,397.166667,190.600000,2.602941,11.8,0.220588,0.433333,0.216667,...,0.0,0.216246,0.006141,0.211904,0.220588,0.008684,-0.008684,0.008684,0.000373,comedy
4,2025-06-05 16:26:04,kenji,60.0,391.000000,189.333333,2.602941,11.8,0.220588,6.295413,2.098471,...,0.0,0.216246,0.006141,0.211904,0.220588,0.008684,-0.008684,0.008684,0.000373,comedy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1255,2025-06-12 10:33:07,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.509902,0.509902,...,0.0,0.231070,0.000000,0.231070,0.231070,0.000000,0.000000,0.000000,0.000000,horror
1256,2025-06-12 10:33:08,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.000000,0.000000,...,0.0,0.231070,0.000000,0.231070,0.231070,0.000000,0.000000,0.000000,0.000000,horror
1257,2025-06-12 10:33:09,aimen,70.0,355.741935,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,...,0.0,0.231070,0.000000,0.231070,0.231070,0.000000,0.000000,0.000000,0.000000,horror
1258,2025-06-12 10:33:10,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,...,0.0,0.231070,0.000000,0.231070,0.231070,0.000000,0.000000,0.000000,0.000000,horror


## 3. Higher-Order Stats for Eye-Tracking


In [36]:
# skewness & kurtosis 
def compute_stats(group):
    return pd.Series({
        'pupil_skew': skew(group['pupil_diameter_mm'], bias=False),
        'pupil_kurtosis': kurtosis(group['pupil_diameter_mm'], bias=False),
        'iris_skew': skew(group['iris_diameter_mm'], bias=False),
        'iris_kurtosis': kurtosis(group['iris_diameter_mm'], bias=False),
        'ratio_skew': skew(group['pupil_iris_ratio'], bias=False),
        'ratio_kurtosis': kurtosis(group['pupil_iris_ratio'], bias=False),
    })

# Step 1: Compute skewness & kurtosis per participant-genre
agg_stats = df.groupby('participant').apply(compute_stats).reset_index()

# Step 2: Merge back with the full original dataframe
df = df.merge(agg_stats, on='participant', how='left')

# Step 3: Reorder columns if needed
df = reorder_cols(df)
df


,timestamp,participant,heart_rate,x_coordinate,y_coordinate,pupil_diameter_mm,iris_diameter_mm,pupil_iris_ratio,head_displacement,head_velocity,...,td_pupil_iris_ratio_slope,td_pupil_iris_ratio_mean_abs_change,td_pupil_iris_ratio_second_order_diff_mean,pupil_skew,pupil_kurtosis,iris_skew,iris_kurtosis,ratio_skew,ratio_kurtosis,genre
0,2025-06-05 16:25:55,kenji,62.0,409.206897,194.000000,2.602941,11.8,0.220588,0.000000,0.000000,...,0.000000,0.000000,0.000373,1.066933,1.049525,NaN,NaN,1.066933,1.049525,comedy
1,2025-06-05 16:25:56,kenji,61.5,413.433333,194.800000,2.602941,11.8,0.220588,4.301484,4.301484,...,0.000000,0.000000,0.000373,1.066933,1.049525,NaN,NaN,1.066933,1.049525,comedy
2,2025-06-05 16:25:59,kenji,60.0,397.333333,191.000000,2.602941,11.8,0.220588,16.542370,5.514123,...,-0.008684,0.008684,0.000373,1.066933,1.049525,NaN,NaN,1.066933,1.049525,comedy
3,2025-06-05 16:26:01,kenji,60.0,397.166667,190.600000,2.602941,11.8,0.220588,0.433333,0.216667,...,-0.008684,0.008684,0.000373,1.066933,1.049525,NaN,NaN,1.066933,1.049525,comedy
4,2025-06-05 16:26:04,kenji,60.0,391.000000,189.333333,2.602941,11.8,0.220588,6.295413,2.098471,...,-0.008684,0.008684,0.000373,1.066933,1.049525,NaN,NaN,1.066933,1.049525,comedy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9351,2025-06-12 10:33:07,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.509902,0.509902,...,0.000000,0.000000,0.000000,1.039181,1.086814,NaN,NaN,1.039181,1.086814,horror
9352,2025-06-12 10:33:08,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.039181,1.086814,NaN,NaN,1.039181,1.086814,horror
9353,2025-06-12 10:33:09,aimen,70.0,355.741935,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,...,0.000000,0.000000,0.000000,1.039181,1.086814,NaN,NaN,1.039181,1.086814,horror
9354,2025-06-12 10:33:10,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,...,0.000000,0.000000,0.000000,1.039181,1.086814,NaN,NaN,1.039181,1.086814,horror


In [37]:
df['iris_skew'] = df['iris_skew'].fillna(0)
df['iris_kurtosis'] = df['iris_kurtosis'].fillna(0)
df

,timestamp,participant,heart_rate,x_coordinate,y_coordinate,pupil_diameter_mm,iris_diameter_mm,pupil_iris_ratio,head_displacement,head_velocity,...,td_pupil_iris_ratio_slope,td_pupil_iris_ratio_mean_abs_change,td_pupil_iris_ratio_second_order_diff_mean,pupil_skew,pupil_kurtosis,iris_skew,iris_kurtosis,ratio_skew,ratio_kurtosis,genre
0,2025-06-05 16:25:55,kenji,62.0,409.206897,194.000000,2.602941,11.8,0.220588,0.000000,0.000000,...,0.000000,0.000000,0.000373,1.066933,1.049525,0.0,0.0,1.066933,1.049525,comedy
1,2025-06-05 16:25:56,kenji,61.5,413.433333,194.800000,2.602941,11.8,0.220588,4.301484,4.301484,...,0.000000,0.000000,0.000373,1.066933,1.049525,0.0,0.0,1.066933,1.049525,comedy
2,2025-06-05 16:25:59,kenji,60.0,397.333333,191.000000,2.602941,11.8,0.220588,16.542370,5.514123,...,-0.008684,0.008684,0.000373,1.066933,1.049525,0.0,0.0,1.066933,1.049525,comedy
3,2025-06-05 16:26:01,kenji,60.0,397.166667,190.600000,2.602941,11.8,0.220588,0.433333,0.216667,...,-0.008684,0.008684,0.000373,1.066933,1.049525,0.0,0.0,1.066933,1.049525,comedy
4,2025-06-05 16:26:04,kenji,60.0,391.000000,189.333333,2.602941,11.8,0.220588,6.295413,2.098471,...,-0.008684,0.008684,0.000373,1.066933,1.049525,0.0,0.0,1.066933,1.049525,comedy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9351,2025-06-12 10:33:07,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.509902,0.509902,...,0.000000,0.000000,0.000000,1.039181,1.086814,0.0,0.0,1.039181,1.086814,horror
9352,2025-06-12 10:33:08,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.039181,1.086814,0.0,0.0,1.039181,1.086814,horror
9353,2025-06-12 10:33:09,aimen,70.0,355.741935,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,...,0.000000,0.000000,0.000000,1.039181,1.086814,0.0,0.0,1.039181,1.086814,horror
9354,2025-06-12 10:33:10,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,...,0.000000,0.000000,0.000000,1.039181,1.086814,0.0,0.0,1.039181,1.086814,horror


## 4. Frequency-Domain Features


In [38]:
def extract_fft_features_multi(signal_dict):
    """Expect a dict of signals {feature_name: np.array}, return flat dict with names."""
    results = {}
    sampling_rate = 1

    for feat_name, signal in signal_dict.items():
        N = len(signal)
        if N < 4:
            # Fill NaNs for all freq features
            results.update({f"{feat_name}_fft_max_amp": np.nan,
                            f"{feat_name}_fft_dominant_freq": np.nan,
                            f"{feat_name}_fft_centroid": np.nan,
                            f"{feat_name}_fft_entropy": np.nan,
                            f"{feat_name}_fft_band_0_0.1": np.nan,
                            f"{feat_name}_fft_band_0.1_0.25": np.nan,
                            f"{feat_name}_fft_band_0.25_0.5": np.nan})
            continue

        freqs = np.fft.fftfreq(N, d=1/sampling_rate)
        fft_values = fft(signal)
        amplitudes = np.abs(fft_values[:N // 2])
        freqs = freqs[:N // 2]

        power = amplitudes**2
        power_sum = np.sum(power)
        power_norm = power / power_sum if power_sum != 0 else np.zeros_like(power)

        results[f"{feat_name}_fft_max_amp"] = np.max(amplitudes)
        results[f"{feat_name}_fft_dominant_freq"] = freqs[np.argmax(amplitudes)]
        results[f"{feat_name}_fft_centroid"] = np.sum(freqs * amplitudes) / np.sum(amplitudes)
        results[f"{feat_name}_fft_entropy"] = entropy(power_norm)
        results[f"{feat_name}_fft_band_0_0.1"] = np.mean(amplitudes[(freqs >= 0.0) & (freqs < 0.1)])
        results[f"{feat_name}_fft_band_0.1_0.25"] = np.mean(amplitudes[(freqs >= 0.1) & (freqs < 0.25)])
        results[f"{feat_name}_fft_band_0.25_0.5"] = np.mean(amplitudes[(freqs >= 0.25) & (freqs <= 0.5)])

    return pd.Series(results)

features_to_process = ['heart_rate', 'pupil_diameter_mm', 'iris_diameter_mm', 'pupil_iris_ratio']

df_freq = (
    df.groupby(['participant'])
      .apply(lambda g: extract_fft_features_multi({feat: g[feat].values for feat in features_to_process}))
      .reset_index()
)

df_freq

,participant,heart_rate_fft_max_amp,heart_rate_fft_dominant_freq,heart_rate_fft_centroid,heart_rate_fft_entropy,heart_rate_fft_band_0_0.1,heart_rate_fft_band_0.1_0.25,heart_rate_fft_band_0.25_0.5,pupil_diameter_mm_fft_max_amp,pupil_diameter_mm_fft_dominant_freq,...,iris_diameter_mm_fft_band_0_0.1,iris_diameter_mm_fft_band_0.1_0.25,iris_diameter_mm_fft_band_0.25_0.5,pupil_iris_ratio_fft_max_amp,pupil_iris_ratio_fft_dominant_freq,pupil_iris_ratio_fft_centroid,pupil_iris_ratio_fft_entropy,pupil_iris_ratio_fft_band_0_0.1,pupil_iris_ratio_fft_band_0.1_0.25,pupil_iris_ratio_fft_band_0.25_0.5
0,aimen,177688.117521,0.0,0.033284,0.017212,997.212189,32.313608,23.132382,4287.568923,0.0,...,117.712195,2.144401e-13,2.133841e-13,363.353299,0.0,0.083466,0.709213,5.949374,0.833584,0.377488
1,clara,302461.500000,0.0,0.020129,0.015491,1018.315914,22.877784,10.466693,6348.239174,0.0,...,117.885437,1.491382e-14,7.929908e-15,537.986371,0.0,0.113664,0.761980,6.670696,1.406232,0.733583
2,kenji,165459.389419,0.0,0.032664,0.033654,969.188820,42.615875,15.316863,5312.958798,0.0,...,117.830824,3.783432e-14,2.027039e-14,450.250746,0.0,0.066454,0.570962,6.533747,0.644215,0.272008


In [39]:
df = df.merge(df_freq, on=['participant'], how='left')
df = reorder_cols(df)
df

,timestamp,participant,heart_rate,x_coordinate,y_coordinate,pupil_diameter_mm,iris_diameter_mm,pupil_iris_ratio,head_displacement,head_velocity,...,iris_diameter_mm_fft_band_0.1_0.25,iris_diameter_mm_fft_band_0.25_0.5,pupil_iris_ratio_fft_max_amp,pupil_iris_ratio_fft_dominant_freq,pupil_iris_ratio_fft_centroid,pupil_iris_ratio_fft_entropy,pupil_iris_ratio_fft_band_0_0.1,pupil_iris_ratio_fft_band_0.1_0.25,pupil_iris_ratio_fft_band_0.25_0.5,genre
0,2025-06-05 16:25:55,kenji,62.0,409.206897,194.000000,2.602941,11.8,0.220588,0.000000,0.000000,...,3.783432e-14,2.027039e-14,450.250746,0.0,0.066454,0.570962,6.533747,0.644215,0.272008,comedy
1,2025-06-05 16:25:56,kenji,61.5,413.433333,194.800000,2.602941,11.8,0.220588,4.301484,4.301484,...,3.783432e-14,2.027039e-14,450.250746,0.0,0.066454,0.570962,6.533747,0.644215,0.272008,comedy
2,2025-06-05 16:25:59,kenji,60.0,397.333333,191.000000,2.602941,11.8,0.220588,16.542370,5.514123,...,3.783432e-14,2.027039e-14,450.250746,0.0,0.066454,0.570962,6.533747,0.644215,0.272008,comedy
3,2025-06-05 16:26:01,kenji,60.0,397.166667,190.600000,2.602941,11.8,0.220588,0.433333,0.216667,...,3.783432e-14,2.027039e-14,450.250746,0.0,0.066454,0.570962,6.533747,0.644215,0.272008,comedy
4,2025-06-05 16:26:04,kenji,60.0,391.000000,189.333333,2.602941,11.8,0.220588,6.295413,2.098471,...,3.783432e-14,2.027039e-14,450.250746,0.0,0.066454,0.570962,6.533747,0.644215,0.272008,comedy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9351,2025-06-12 10:33:07,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.509902,0.509902,...,2.144401e-13,2.133841e-13,363.353299,0.0,0.083466,0.709213,5.949374,0.833584,0.377488,horror
9352,2025-06-12 10:33:08,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.000000,0.000000,...,2.144401e-13,2.133841e-13,363.353299,0.0,0.083466,0.709213,5.949374,0.833584,0.377488,horror
9353,2025-06-12 10:33:09,aimen,70.0,355.741935,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,...,2.144401e-13,2.133841e-13,363.353299,0.0,0.083466,0.709213,5.949374,0.833584,0.377488,horror
9354,2025-06-12 10:33:10,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,...,2.144401e-13,2.133841e-13,363.353299,0.0,0.083466,0.709213,5.949374,0.833584,0.377488,horror


## 5. Nonlinear Irregularity 

In [40]:
# For heart_rate, pupil_diameter_mm: approximate_entropy, sample_entropy

def _phi(x, m, r):
    N = len(x)
    x_m = np.array([x[i:i+m] for i in range(N - m + 1)])
    C = np.sum(np.max(np.abs(x_m[:, None] - x_m[None, :]), axis=2) <= r, axis=1) / (N - m + 1)
    return np.sum(np.log(C)) / (N - m + 1)

def approximate_entropy(x, m=2, r=None):
    if r is None:
        r = 0.2 * np.std(x)
    return abs(_phi(x, m, r) - _phi(x, m + 1, r))

def sample_entropy(x, m=2, r=None):
    if r is None:
        r = 0.2 * np.std(x)
    N = len(x)
    x_m = np.array([x[i:i+m] for i in range(N - m + 1)])
    x_m1 = np.array([x[i:i+m+1] for i in range(N - m)])
    def _count_similar(template, data, r):
        return np.sum(np.max(np.abs(data - template), axis=1) <= r) - 1  # exclude self-match
    B = np.array([_count_similar(template, x_m, r) for template in x_m])
    A = np.array([_count_similar(template, x_m1, r) for template in x_m1])
    B_sum = np.sum(B)
    A_sum = np.sum(A)
    if B_sum == 0:
        return np.nan
    return -np.log(A_sum / B_sum)

# Features to compute nonlinear irregularity for:
nonlinear_features = ['heart_rate', 'pupil_diameter_mm']

def compute_nonlinear_features(group):
    results = {}
    for feat in nonlinear_features:
        signal = group[feat].values
        if len(signal) < 10:  # or any threshold to ensure meaningful entropy
            results[f"{feat}_approx_entropy"] = np.nan
            results[f"{feat}_sample_entropy"] = np.nan
        else:
            results[f"{feat}_approx_entropy"] = approximate_entropy(signal)
            results[f"{feat}_sample_entropy"] = sample_entropy(signal)
    return pd.Series(results)

# Apply per participant and genre
df_nonlinear = df.groupby(['participant']).apply(compute_nonlinear_features).reset_index()
df_nonlinear

,participant,heart_rate_approx_entropy,heart_rate_sample_entropy,pupil_diameter_mm_approx_entropy,pupil_diameter_mm_sample_entropy
0,aimen,0.684174,0.453317,0.393863,0.205089
1,clara,0.385330,0.322031,0.485169,0.242194
2,kenji,0.615392,0.467991,0.370674,0.195699


In [41]:
df = df.merge(df_nonlinear, on=['participant'], how='left')
df = reorder_cols(df)
df

,timestamp,participant,heart_rate,x_coordinate,y_coordinate,pupil_diameter_mm,iris_diameter_mm,pupil_iris_ratio,head_displacement,head_velocity,...,pupil_iris_ratio_fft_centroid,pupil_iris_ratio_fft_entropy,pupil_iris_ratio_fft_band_0_0.1,pupil_iris_ratio_fft_band_0.1_0.25,pupil_iris_ratio_fft_band_0.25_0.5,heart_rate_approx_entropy,heart_rate_sample_entropy,pupil_diameter_mm_approx_entropy,pupil_diameter_mm_sample_entropy,genre
0,2025-06-05 16:25:55,kenji,62.0,409.206897,194.000000,2.602941,11.8,0.220588,0.000000,0.000000,...,0.066454,0.570962,6.533747,0.644215,0.272008,0.615392,0.467991,0.370674,0.195699,comedy
1,2025-06-05 16:25:56,kenji,61.5,413.433333,194.800000,2.602941,11.8,0.220588,4.301484,4.301484,...,0.066454,0.570962,6.533747,0.644215,0.272008,0.615392,0.467991,0.370674,0.195699,comedy
2,2025-06-05 16:25:59,kenji,60.0,397.333333,191.000000,2.602941,11.8,0.220588,16.542370,5.514123,...,0.066454,0.570962,6.533747,0.644215,0.272008,0.615392,0.467991,0.370674,0.195699,comedy
3,2025-06-05 16:26:01,kenji,60.0,397.166667,190.600000,2.602941,11.8,0.220588,0.433333,0.216667,...,0.066454,0.570962,6.533747,0.644215,0.272008,0.615392,0.467991,0.370674,0.195699,comedy
4,2025-06-05 16:26:04,kenji,60.0,391.000000,189.333333,2.602941,11.8,0.220588,6.295413,2.098471,...,0.066454,0.570962,6.533747,0.644215,0.272008,0.615392,0.467991,0.370674,0.195699,comedy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9351,2025-06-12 10:33:07,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.509902,0.509902,...,0.083466,0.709213,5.949374,0.833584,0.377488,0.684174,0.453317,0.393863,0.205089,horror
9352,2025-06-12 10:33:08,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.000000,0.000000,...,0.083466,0.709213,5.949374,0.833584,0.377488,0.684174,0.453317,0.393863,0.205089,horror
9353,2025-06-12 10:33:09,aimen,70.0,355.741935,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,...,0.083466,0.709213,5.949374,0.833584,0.377488,0.684174,0.453317,0.393863,0.205089,horror
9354,2025-06-12 10:33:10,aimen,70.0,356.000000,324.000000,2.726626,11.8,0.231070,0.258065,0.258065,...,0.083466,0.709213,5.949374,0.833584,0.377488,0.684174,0.453317,0.393863,0.205089,horror


## 6. Categorical + Temporal Patterns


In [42]:
# 1. Discretize each numerical feature per participant using z-score buckets
def discretize_zscore(group, cols):
    # Compute mean and std per participant per feature
    means = group[cols].mean()
    stds = group[cols].std()

    def bucketize(x, mean, std):
        if std == 0 or np.isnan(std):
            return 'normal'  # fallback if no variation
        z = (x - mean) / std
        if z < -1:
            return 'low'
        elif z > 1:
            return 'high'
        else:
            return 'normal'

    for col in cols:
        mean = means[col]
        std = stds[col]
        group[f'{col}_bucket'] = group[col].apply(bucketize, args=(mean, std))

    return group

numerical_cols = ['heart_rate', 'pupil_diameter_mm', 'iris_diameter_mm', 'pupil_iris_ratio']
df = df.groupby(['participant']).apply(lambda g: discretize_zscore(g, numerical_cols))
df = df.reset_index(drop=True)


# 2. Extract temporal succession patterns of these buckets
def create_bigrams(group, col):
    group = group.sort_values('timestamp')
    buckets = group[f'{col}_bucket'].values
    bigrams = [f'{buckets[i]}_{buckets[i+1]}' for i in range(len(buckets)-1)]
    # Align bigrams with timestamps (dropping the last row without a bigram)
    group = group.iloc[:-1].copy()
    group[f'{col}_bigram'] = bigrams
    return group

for col in numerical_cols:
    df = df.groupby(['participant']).apply(lambda g: create_bigrams(g, col))
    df = df.reset_index(drop=True)
    
# 3. Count frequencies or occurrences of these patterns
pattern_counts = {}

for col in numerical_cols:
    counts = (
        df.groupby(['participant'])[f'{col}_bigram']
          .value_counts()
          .unstack(fill_value=0)
          .add_prefix(f'{col}_bigram_')
    )
    pattern_counts[col] = counts

# Combine all pattern counts into one DataFrame:
from functools import reduce
pattern_features = reduce(lambda a, b: a.join(b, how='outer'), pattern_counts.values()).fillna(0).reset_index()


In [43]:
df = df.merge(pattern_features, on=['participant'], how='left')

In [44]:
df.drop(['heart_rate_bigram', 'pupil_diameter_mm_bigram', 'iris_diameter_mm_bigram', 'pupil_iris_ratio_bigram'], axis=1, inplace=True)
df.columns = [col.replace('_bigram', '') for col in df.columns]
df = reorder_cols(df)
df

,timestamp,participant,heart_rate,x_coordinate,y_coordinate,pupil_diameter_mm,iris_diameter_mm,pupil_iris_ratio,head_displacement,head_velocity,...,iris_diameter_mm_normal_normal,pupil_iris_ratio_high_high,pupil_iris_ratio_high_normal,pupil_iris_ratio_low_high,pupil_iris_ratio_low_low,pupil_iris_ratio_low_normal,pupil_iris_ratio_normal_high,pupil_iris_ratio_normal_low,pupil_iris_ratio_normal_normal,genre
6562,2025-06-05 16:25:55,kenji,62.000000,409.206897,194.000000,2.602941,11.8,0.220588,0.000000,0.000000,...,2782,464,27,1,288,26,26,27,1923,comedy
6563,2025-06-05 16:25:56,kenji,61.500000,413.433333,194.800000,2.602941,11.8,0.220588,4.301484,4.301484,...,2782,464,27,1,288,26,26,27,1923,comedy
6564,2025-06-05 16:25:59,kenji,60.000000,397.333333,191.000000,2.602941,11.8,0.220588,16.542370,5.514123,...,2782,464,27,1,288,26,26,27,1923,comedy
6565,2025-06-05 16:26:01,kenji,60.000000,397.166667,190.600000,2.602941,11.8,0.220588,0.433333,0.216667,...,2782,464,27,1,288,26,26,27,1923,comedy
6566,2025-06-05 16:26:04,kenji,60.000000,391.000000,189.333333,2.602941,11.8,0.220588,6.295413,2.098471,...,2782,464,27,1,288,26,26,27,1923,comedy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2445,2025-06-12 10:33:03,aimen,69.555556,354.483871,323.258065,2.726626,11.8,0.231070,0.965066,0.965066,...,2450,444,28,0,401,29,29,29,1490,horror
2446,2025-06-12 10:33:04,aimen,69.666667,354.600000,324.000000,2.726626,11.8,0.231070,0.750969,0.750969,...,2450,444,28,0,401,29,29,29,1490,horror
2447,2025-06-12 10:33:05,aimen,69.777778,355.000000,323.931034,2.726626,11.8,0.231070,0.405902,0.405902,...,2450,444,28,0,401,29,29,29,1490,horror
2448,2025-06-12 10:33:06,aimen,69.888889,355.500000,323.900000,2.726626,11.8,0.231070,0.500962,0.500962,...,2450,444,28,0,401,29,29,29,1490,horror


## 7. Per-Person Normalization 

In [45]:
# Leave for modelling

## 8. Clustering?

In [46]:
# not necessarily?

## Final Step: save final set of features

In [47]:
df.to_csv('data_with_new_features_v2.csv', index=False)

In [48]:
with open("columns.txt", "w") as f:
    for col in df.columns:
        f.write(col + "\n")
